In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATA_DIR = "/content/drive/MyDrive/malaria_cnn_project/tfds_data"

In [ ]:
!pip install -q --upgrade \
    "protobuf>=6.31.1" \
    tensorflow-metadata \
    tensorflow-datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.


In [ ]:
import google.protobuf

print("Protobuf:", google.protobuf.__version__)

Protobuf: 5.29.6


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [ ]:
!pip install -q --upgrade \
    "protobuf==6.31.1" \
    "tensorflow-datasets==4.9.10" \
    tensorflow-metadata

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import google.protobuf

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("Protobuf:", google.protobuf.__version__)

/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations in the next r

TensorFlow: 2.20.0
TFDS: 4.9.10
Protobuf: 6.31.1


In [ ]:
(train_dataset,
 val_dataset,
 test_dataset), dataset_info = tfds.load(

    "malaria",

    split=[
        "train[:80%]",
        "train[80%:90%]",
        "train[90%:]"
    ],

    shuffle_files=True,
    with_info=True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/malaria/incomplete.O3CSY6_1.0.0/malaria-train.tfrecord-[0-9][0-9][0-9][0-9…

Dataset malaria downloaded and prepared to /root/tensorflow_datasets/malaria/1.0.0. Subsequent calls will reuse this data.


# Processing করার আগে raw image inspect করো

In [ ]:
for sample in train_dataset.take(1):

    image = sample["image"]
    label = sample["label"]

    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)
    print("Label:", label.numpy())

Image shape: (145, 148, 3)
Image dtype: <dtype: 'uint8'>
Label: 1


# Resize images: feature scaling so that oscillation never occurs ,and one features values large another one small

In [ ]:
IM_SIZE = 224

for sample in train_dataset.take(1):

    image = sample["image"]
    label = sample["label"]

    print("Before resize:", image.shape)

    resized_image = tf.image.resize(
        image,
        (IM_SIZE, IM_SIZE)
    )

    print("After resize:", resized_image.shape)

Before resize: (103, 103, 3)
After resize: (224, 224, 3)


# Before normalization, first inspect the resized image values:

In [ ]:
print("Dtype:", resized_image.dtype)
print("Min pixel:", tf.reduce_min(resized_image).numpy())
print("Max pixel:", tf.reduce_max(resized_image).numpy())

Dtype: <dtype: 'float32'>
Min pixel: 0.0
Max pixel: 217.0


In [ ]:
normalized_image = resized_image / 255.0

In [ ]:
normalized_image.numpy()

array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       ...,

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]], dtype=float32)

In [ ]:
normalized_image=resized_image/255.0

In [ ]:
print("shape",normalized_image.shape)

shape (224, 224, 3)


In [ ]:
print("data type",normalized_image.dtype)

data type <dtype: 'float32'>


In [ ]:
print("min pixel",tf.reduce_min(normalized_image).numpy())

min pixel 0.0


In [ ]:
print("min pixel",tf.reduce_max(normalized_image).numpy())

min pixel 0.8509804


In [ ]:
normalized_image = resized_image / 255.0

print("Shape:", normalized_image.shape)
print("Dtype:", normalized_image.dtype)

print("Min pixel:", tf.reduce_min(normalized_image).numpy())
print("Max pixel:", tf.reduce_max(normalized_image).numpy())

Shape: (224, 224, 3)
Dtype: <dtype: 'float32'>
Min pixel: 0.0
Max pixel: 0.8509804


In [ ]:
print("Shape:", normalized_image.shape)

print("Min value:",
      tf.reduce_min(normalized_image).numpy())

print("Max value:",
      tf.reduce_max(normalized_image).numpy())

Shape: (224, 224, 3)
Min value: 0.0
Max value: 0.8509804


# resize_rescale function()

In [ ]:
im_size=224

def resize_rescale(sample):
  # feature or each sample ,we are apply features scaling on images as all img size not same value lager than label value

  image=sample["image"]
  label=sample["label"]

  # resize images
  image=tf.image.resize(image,(im_size,im_size))


  # after resizing devide by 255 so that value btw 0 to 1


  image=image/255.0


  return image ,label

# এখন শুধু ONE sample দিয়ে function test করো

In [ ]:
for sample in train_dataset.take(1):

    image, label = resize_rescale(sample)

    print("Shape:", image.shape)
    print("Label:", label.numpy())
    print("Min:", tf.reduce_min(image).numpy())
    print("Max:", tf.reduce_max(image).numpy())

Shape: (224, 224, 3)
Label: 1
Min: 0.0
Max: 0.7877451


# 🗺️ TensorFlow Dataset `.map()`

## 💡 Main Idea

`.map()` মানে হলো:

> **Dataset-এর প্রত্যেকটি element/sample-এর উপর একই function apply করা।**

---

## 🔄 Visual Flow

```text
Original Dataset
      │
      ▼
┌─────────────┐
│  sample 1   │ ───► resize_rescale() ───► ✅ processed sample 1
├─────────────┤
│  sample 2   │ ───► resize_rescale() ───► ✅ processed sample 2
├─────────────┤
│  sample 3   │ ───► resize_rescale() ───► ✅ processed sample 3
├─────────────┤
│     ...     │
└─────────────┘

# 📦 What is `train_dataset` in TensorFlow?

## ✅ Main Idea

`train_dataset` কোনো **array না** এবং এটা কোনো **single tensor-ও না**।

এটা হলো:

```python
tf.data.Dataset
```

অর্থাৎ TensorFlow-এর একটি **Dataset Object**।

---

## 🧠 Conceptually কী আছে এর ভিতরে?

যখন আমরা লিখি:

```python
train_dataset = dataset["train"]
```

তখন `train_dataset`-এর ভিতরে অনেকগুলো **sample** থাকে।

```text
train_dataset
│
├── 🖼️ Sample 1
│    ├── image  → Tensor
│    └── label  → Tensor
│
├── 🖼️ Sample 2
│    ├── image  → Tensor
│    └── label  → Tensor
│
├── 🖼️ Sample 3
│    ├── image  → Tensor
│    └── label  → Tensor
│
├── 🖼️ Sample 4
│    ├── image  → Tensor
│    └── label  → Tensor
│
└── ...
```

---

# 🔍 Dataset vs Sample vs Tensor

```text
train_dataset
     │
     │ contains many
     ▼
   Samples
     │
     ├── image ───► Tensor
     │
     └── label ───► Tensor
```

### তাই:

```text
train_dataset ≠ Tensor
train_dataset ≠ Array

train_dataset = Collection / Pipeline of Samples
```

কিন্তু প্রতিটি sample-এর ভিতরের:

```text
image → Tensor
label → Tensor
```

---

# 📌 Example

একটি sample দেখার জন্য:

```python
for sample in train_dataset.take(1):
    print(sample)
```

Conceptually output:

```text
{
    "image": <Tensor ...>,
    "label": <Tensor ...>
}
```

অর্থাৎ:

```text
sample
│
├── image
│     └── Tensor
│
└── label
      └── Tensor
```

---

# 🖼️ Image Tensor

একটি image এমন হতে পারে:

```python
sample["image"]
```

Shape:

```text
(height, width, channels)
```

Example:

```text
(148, 142, 3)
```

এখানে:

```text
148 → height
142 → width
3   → RGB channels
```

---

# 🏷️ Label Tensor

```python
sample["label"]
```

এটাও Tensor।

Example:

```text
tf.Tensor(0, shape=(), dtype=int64)
```

এখানে:

```text
0 → class label
```

Malaria dataset-এর ক্ষেত্রে conceptually:

```text
0 → parasitized
1 → uninfected
```

---

# 🔄 `.map()` কেন Dataset-এর উপর কাজ করে?

কারণ `train_dataset`-এর ভিতরে অনেক sample আছে।

যখন লিখি:

```python
train_dataset = train_dataset.map(resize_rescale)
```

তখন TensorFlow করে:

```text
train_dataset
│
├── sample 1 ──► resize_rescale()
│
├── sample 2 ──► resize_rescale()
│
├── sample 3 ──► resize_rescale()
│
└── ...
```

প্রতিটি sample-এর:

```text
image → resize + normalize
label → unchanged
```

---

# 🎯 Final Mental Model

```text
tf.data.Dataset
      │
      │
      ├── Sample
      │     ├── Image Tensor
      │     └── Label Tensor
      │
      ├── Sample
      │     ├── Image Tensor
      │     └── Label Tensor
      │
      └── ...
```

## ⭐ Remember

> `train_dataset` = অনেক sample বহন করা TensorFlow Dataset object।

আর,

> `sample["image"]` এবং `sample["label"]` = Tensor।

### এক লাইনে:

```text
Dataset → Samples → Tensors
```


# how  to apply resize_rescale function on all sample of training dataset

In [ ]:
train_processed = train_dataset.map(
    resize_rescale
)

In [ ]:
train_dataset

<_PrefetchDataset element_spec={'image': TensorSpec(shape=(None, None, 3), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>

In [ ]:
train_processed = train_dataset.map(resize_rescale)

print(type(train_dataset))
print(type(train_processed))

<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
<class 'tensorflow.python.data.ops.map_op._MapDataset'>


In [ ]:
for image, label in train_processed.take(1):

    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)

    print("Label:", label.numpy())

    print("Min pixel:", tf.reduce_min(image).numpy())
    print("Max pixel:", tf.reduce_max(image).numpy())

Image shape: (224, 224, 3)
Image dtype: <dtype: 'float32'>
Label: 1
Min pixel: 0.0
Max pixel: 0.8751619


# next step: validation আর test dataset-এও same preprocessing apply করো।

In [ ]:
val_processed=val_dataset.map(resize_rescale)

In [ ]:
test_processed=test_dataset.map(resize_rescale)